In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/NLP/NLP Ass3/SMSSpamCollection"

df_sms = pd.read_csv(
    file_path,
    sep="\t",
    header=None,
    names=["label", "text"],
    encoding="utf-8"
)

df_sms.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
df_sms.shape

(5572, 2)

In [ ]:
df_sms["label"].value_counts()

,count
label,
ham,4825
spam,747


In [ ]:
df_sms["label"].value_counts(normalize=True) * 100

,proportion
label,
ham,86.593683
spam,13.406317


In [ ]:
df_sms.isnull().sum()

,0
label,0
text,0


In [ ]:
df_sms.duplicated(subset=["text"]).sum()

np.int64(403)

In [ ]:
df_sms[df_sms.duplicated(subset=["text"], keep=False)].sort_values("text").head(20)

,label,text
505,spam,+123 Congratulations - in this week's competit...
2124,spam,+123 Congratulations - in this week's competit...
2344,ham,1) Go to write msg 2) Put on Dictionary mode 3...
1373,ham,1) Go to write msg 2) Put on Dictionary mode 3...
2163,ham,1) Go to write msg 2) Put on Dictionary mode 3...
1050,spam,18 days to Euro2004 kickoff! U will be kept in...
2719,spam,18 days to Euro2004 kickoff! U will be kept in...
2044,spam,4mths half price Orange line rental & latest c...
389,spam,4mths half price Orange line rental & latest c...
1779,ham,7 wonders in My WORLD 7th You 6th Ur style 5th...


In [ ]:
df_sms_clean = df_sms.drop_duplicates(subset=["text"], keep="first").reset_index(drop=True)
print("Before removing duplicates:", df_sms.shape)
print("After removing duplicates:", df_sms_clean.shape)
print("Removed rows:", df_sms.shape[0] - df_sms_clean.shape[0])
print("Remaining duplicates:", df_sms_clean.duplicated(subset=["text"]).sum())

Before removing duplicates: (5572, 2)
After removing duplicates: (5169, 2)
Removed rows: 403
Remaining duplicates: 0


In [ ]:
df_sms_clean["label"].value_counts()

,count
label,
ham,4516
spam,653


In [ ]:
non_ascii = df_sms_clean[df_sms_clean["text"].str.contains(r"[^\x00-\x7F]", na=False)]

print("Non-ASCII rows:", non_ascii.shape[0])
non_ascii[["label", "text"]].head(30)

Non-ASCII rows: 443


,label,text
5,spam,FreeMsg Hey there darling it's been 3 week's n...
8,spam,WINNER!! As a valued network customer you have...
12,spam,URGENT! You have won a 1 week FREE membership ...
18,ham,Fine if thats the way u feel. Thats the way ...
19,spam,England v Macedonia - dont miss the goals/team...
21,ham,I‘m going to try for 2 months ha ha only joking
22,ham,So ü pay first lar... Then when is da stock co...
34,spam,Thanks for your subscription to Ringtone UK yo...
35,ham,Yup... Ok i go home look at the timings then i...
65,spam,"As a valued customer, I am pleased to advise y..."


In [ ]:
df_sms_clean["label_binary"] = df_sms_clean["label"].map({
    "ham": 0,
    "spam": 1
})

df_sms_clean[["label", "label_binary", "text"]].head()


,label,label_binary,text
0,ham,0,"Go until jurong point, crazy.. Available only ..."
1,ham,0,Ok lar... Joking wif u oni...
2,spam,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,0,U dun say so early hor... U c already then say...
4,ham,0,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
df_sms_clean["label_binary"].isnull().sum()

np.int64(0)

In [ ]:
df_sms_clean.head()

,label,text,label_binary
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [ ]:
print(df_sms_clean.shape)
print(df_sms_clean.isnull().sum())
print(df_sms_clean["label"].value_counts())
print(df_sms_clean["label_binary"].value_counts())

(5169, 3)
label           0
text            0
label_binary    0
dtype: int64
label
ham     4516
spam     653
Name: count, dtype: int64
label_binary
0    4516
1     653
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df_sms_clean,
    test_size=0.30,
    random_state=42,
    stratify=df_sms_clean["label_binary"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label_binary"]
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (3618, 3)
Validation: (775, 3)
Test: (776, 3)


In [ ]:
print("Train label distribution:")
print(train_df["label"].value_counts())
print(train_df["label_binary"].value_counts(normalize=True) * 100)

print("\nValidation label distribution:")
print(val_df["label"].value_counts())
print(val_df["label_binary"].value_counts(normalize=True) * 100)

print("\nTest label distribution:")
print(test_df["label"].value_counts())
print(test_df["label_binary"].value_counts(normalize=True) * 100)

Train label distribution:
label
ham     3161
spam     457
Name: count, dtype: int64
label_binary
0    87.368712
1    12.631288
Name: proportion, dtype: float64

Validation label distribution:
label
ham     677
spam     98
Name: count, dtype: int64
label_binary
0    87.354839
1    12.645161
Name: proportion, dtype: float64

Test label distribution:
label
ham     678
spam     98
Name: count, dtype: int64
label_binary
0    87.371134
1    12.628866
Name: proportion, dtype: float64


In [ ]:
import os

output_folder = "/content/drive/MyDrive/NLP/NLP Ass3/processed_data_v1"
os.makedirs(output_folder, exist_ok=True)

In [ ]:
df_sms_clean.to_csv(f"{output_folder}/cleaned_sms_spam.csv", index=False)
train_df.to_csv(f"{output_folder}/train.csv", index=False)
val_df.to_csv(f"{output_folder}/val.csv", index=False)
test_df.to_csv(f"{output_folder}/test.csv", index=False)

In [ ]:
os.listdir(output_folder)

['cleaned_sms_spam.csv', 'train.csv', 'val.csv', 'test.csv']